### Imports and data load:

In [54]:
import pandas as pd 
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
import optuna
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import cross_val_score
import mlflow
import mlflow.sklearn
import dagshub
from catboost import CatBoostRegressor
import json

In [48]:
mlflow.set_tracking_uri("https://dagshub.com/PriyanshuMewal/delivery-time-prediction.mlflow")

dagshub.init(repo_owner='PriyanshuMewal', repo_name='delivery-time-prediction', mlflow=True)

mlflow.set_experiment("Experiment 5: HP tunning over top contenders")

Initialized MLflow to track repo "PriyanshuMewal/delivery-time-prediction"

Repository PriyanshuMewal/delivery-time-prediction initialized!

2026/02/20 17:59:21 INFO mlflow.tracking.fluent: Experiment with name 'Experiment 5: HP tunning over top contenders' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/b23b4ef0857f41d4acc3aa6d315efae7', creation_time=1771590667580, experiment_id='7', last_update_time=1771590667580, lifecycle_stage='active', name='Experiment 5: HP tunning over top contenders', tags={}>

In [2]:
train_df = pd.read_csv("../data/processed/with_imputation/train_df.csv")
test_df = pd.read_csv("../data/processed/with_imputation/test_df.csv")

In [4]:
X_train = train_df.drop(columns=["time"])
X_test= test_df.drop(columns=["time"])

y_train = train_df["time"]
y_test = test_df["time"]

### Hyperparameter tunning over best candidates:

In [45]:
def objective(trial, model):

    if model().__class__.__name__ == XGBRegressor().__class__.__name__:

        params = {
        "n_estimators": trial.suggest_int("n_estimators", 500, 3000),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "min_child_weight": trial.suggest_float("min_child_weight", 1, 20),
        "gamma": trial.suggest_float("gamma", 0, 5),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10, log=True),
        "tree_method": "hist",
        "objective": "reg:squarederror"
        }

    elif model().__class__.__name__ == LGBMRegressor().__class__.__name__:
        
        params = {
            "n_estimators": trial.suggest_int("n_estimators", 500, 3000),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
            "num_leaves": trial.suggest_int("num_leaves", 16, 256),
            "max_depth": trial.suggest_int("max_depth", 3, 12),
            "min_child_samples": trial.suggest_int("min_child_samples", 5, 100),
            "feature_fraction": trial.suggest_float("feature_fraction", 0.6, 1.0),
            "bagging_fraction": trial.suggest_float("bagging_fraction", 0.6, 1.0),
            "lambda_l1": trial.suggest_float("lambda_l1", 1e-8, 10, log=True),
            "lambda_l2": trial.suggest_float("lambda_l2", 1e-8, 10, log=True),
            "objective": "regression",
            "verbosity": -1
        }

    else:
        params = {
            "iterations": trial.suggest_int("iterations", 500, 3000),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
            "depth": trial.suggest_int("depth", 4, 10),
            "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1, 10, log=True),
            "min_data_in_leaf": trial.suggest_int("min_data_in_leaf", 1, 100),
            "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 1.0),
            "loss_function": "RMSE",
            "verbose": False
        }

    # Train the model:
    model = model(**params)

    model.fit(X_train, y_train)

    # Evaluate model:
    y_pred = model.predict(X_test)

    # calculate mae, r2_score:
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    trial.set_user_attr("r2", r2)
    return mae

#### Evaluate hyperparameter tunning for XGBoost:

In [22]:
best_params = {}
best_values = {}

In [46]:
study = optuna.create_study(study_name="xgboost_study", direction="minimize")
study.optimize(lambda trial: objective(trial, XGBRegressor), n_trials=50)

best_params.update({"XGboost": study.best_params})
best_values.update({"XGboost": {"mean_absolute_error": study.best_value,
                                "r2_score": study.best_trial.user_attrs["r2"]}})

[I 2026-02-20 17:47:56,341] A new study created in memory with name: xgboost_study
[I 2026-02-20 17:48:03,425] Trial 0 finished with value: 0.3401850627168619 and parameters: {'n_estimators': 1806, 'learning_rate': 0.2112148840853212, 'max_depth': 8, 'min_child_weight': 2.6759649138446244, 'gamma': 1.5562002689200454, 'subsample': 0.620659774315707, 'colsample_bytree': 0.9492539598963894, 'reg_alpha': 0.014009051938483124, 'reg_lambda': 3.170161330149263e-05}. Best is trial 0 with value: 0.3401850627168619.
[I 2026-02-20 17:48:14,372] Trial 1 finished with value: 0.3439601676104897 and parameters: {'n_estimators': 1531, 'learning_rate': 0.07463585607591847, 'max_depth': 7, 'min_child_weight': 14.44019250979094, 'gamma': 0.05122867416544663, 'subsample': 0.8118557721934447, 'colsample_bytree': 0.635059869453259, 'reg_alpha': 0.5260943681585277, 'reg_lambda': 7.453524635929396}. Best is trial 0 with value: 0.3401850627168619.
[I 2026-02-20 17:48:21,230] Trial 2 finished with value: 0.373

#### Evaluate hyperparameter tunning for LGBRegressor:

In [23]:
study = optuna.create_study(study_name="lgbm_study", direction="minimize")
study.optimize(lambda trial: objective(trial, LGBMRegressor), n_trials=50)

best_params.update({"lightgbm": study.best_params})

[I 2026-02-20 16:29:52,440] A new study created in memory with name: lgbm_study
[I 2026-02-20 16:29:53,838] Trial 0 finished with value: 0.33969495495894303 and parameters: {'n_estimators': 508, 'learning_rate': 0.1900721108221585, 'num_leaves': 20, 'max_depth': 11, 'min_child_samples': 38, 'feature_fraction': 0.8530630938286046, 'bagging_fraction': 0.7617668485946705, 'lambda_l1': 2.0041610672166232, 'lambda_l2': 1.22755529566029e-07}. Best is trial 0 with value: 0.33969495495894303.
[I 2026-02-20 16:29:58,038] Trial 1 finished with value: 0.37275882934679627 and parameters: {'n_estimators': 1982, 'learning_rate': 0.015478865041109542, 'num_leaves': 100, 'max_depth': 3, 'min_child_samples': 68, 'feature_fraction': 0.9595472380423566, 'bagging_fraction': 0.8015409100890637, 'lambda_l1': 1.2772242486755893e-08, 'lambda_l2': 5.494963494759092e-05}. Best is trial 0 with value: 0.33969495495894303.
[I 2026-02-20 16:30:01,896] Trial 2 finished with value: 0.33355534543184123 and parameters:

AttributeError: 'Study' object has no attribute 'best_values'

In [28]:
best_values.update({"lightgbm": {"mean_absolute_error": study.best_value,
                                "r2_score": study.best_trial.user_attrs["r2"]}})

#### Evaluate hyperparameter tunning for Catboost:

In [ ]:
study = optuna.create_study(study_name="catboost_study", direction="minimize")
study.optimize(lambda trial: objective(trial, CatBoostRegressor), n_trials=50)

best_params.update({"catboost": study.best_params})
best_values.update({"catboost": {"mean_absolute_error": study.best_value,
                                "r2_score": study.best_trial.user_attrs["r2"]}})

In [59]:
best_params.update({"catboost": {"iterations": 1658, "learning_rate": 0.012937919335703926,
                                 "depth": 10, "l2_leaf_reg": 1.6558560841722005, 
                                 "loss_function": "RMSE", "bagging_temperature": 0.9829734460508931,
                                 "min_data_in_leaf": 64}})

with open("../reports/best_params.json", "w") as file:
    json.dump(best_params, file, indent=4)

In [60]:
best_params

{'lightgbm': {'n_estimators': 1043,
  'learning_rate': 0.01184348616544254,
  'num_leaves': 137,
  'max_depth': 11,
  'min_child_samples': 23,
  'feature_fraction': 0.7839333790779994,
  'bagging_fraction': 0.9697068541421031,
  'lambda_l1': 3.972979937157174e-08,
  'lambda_l2': 2.639574087334251e-08},
 'XGboost': {'n_estimators': 503,
  'learning_rate': 0.016063959766841125,
  'max_depth': 10,
  'min_child_weight': 3.6328861966907344,
  'gamma': 0.19546838376630365,
  'subsample': 0.85129326702622,
  'colsample_bytree': 0.9778384789059209,
  'reg_alpha': 0.011603836281718623,
  'reg_lambda': 1.2982059192370237e-05},
 'catboost': {'iterations': 1658,
  'learning_rate': 0.012937919335703926,
  'depth': 10,
  'l2_leaf_reg': 1.6558560841722005,
  'loss_function': 'RMSE',
  'bagging_temperature': 0.9829734460508931,
  'min_data_in_leaf': 64}}

#### Log experiments over mlflow:

In [49]:
def log_mlflow(model, X_train, y_train, X_test, y_test):

    with mlflow.start_run(run_name=model.__class__.__name__):
        
        parameters = {"model": model.__class__.__name__}
        
        # log parameters:
        parameters.update(model.get_params())
        mlflow.log_params(parameters)
    
        # train and evaluate model:
        model.fit(X_train, y_train)
    
        # mae and r2_score for training and testing and also cross_val_score:
        y_pred_train = model.predict(X_train)
        y_pred = model.predict(X_test)
        
        metrics = {"mae_train": mean_absolute_error(y_train, y_pred_train),
                   "mae_test": mean_absolute_error(y_test, y_pred), 
                   "r2_train": r2_score(y_train, y_pred_train),
                   "r2_test": r2_score(y_test, y_pred)}
    
        scores = cross_val_score(model, X_train, y_train, cv=5, scoring="neg_mean_absolute_error")
        metrics.update({"neg_mean_absolute_error": scores.mean()})
    
        mlflow.log_metrics(metrics)
    
        # log model:
        signature = mlflow.models.infer_signature(X_test, y_pred)
        mlflow.sklearn.log_model(model, signature=signature)
    
        # set artifact:
        mlflow.log_artifact("../data/processed/with_imputation/train_df.csv")
        mlflow.log_artifact("../data/processed/with_imputation/test_df.csv")

In [50]:
models = ["xgb", "lightgbm", "catboost"]
for model in models:
    if model == "xgb":
        log_mlflow(XGBRegressor(**best_params["XGboost"]), X_train, y_train, X_test, y_test)
    elif model == "lightgbm":
        log_mlflow(LGBMRegressor(**best_params["lightgbm"]), X_train, y_train, X_test, y_test)
    elif model == "catboost":
        log_mlflow(CatBoostRegressor(**best_params["catboost"]), X_train, y_train, X_test, y_test)

D:\Andacoda\Lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


🏃 View run XGBRegressor at: https://dagshub.com/PriyanshuMewal/delivery-time-prediction.mlflow/#/experiments/7/runs/9e958b5b6507491dadf6b00313170911
🧪 View experiment at: https://dagshub.com/PriyanshuMewal/delivery-time-prediction.mlflow/#/experiments/7


D:\Andacoda\Lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


🏃 View run LGBMRegressor at: https://dagshub.com/PriyanshuMewal/delivery-time-prediction.mlflow/#/experiments/7/runs/c469d0dd0f294aeab996bb465475ff03
🧪 View experiment at: https://dagshub.com/PriyanshuMewal/delivery-time-prediction.mlflow/#/experiments/7
